In [2]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

/var/folders/wk/tl2drvdn73s_lt_qcp_4ksx40000gn/T/ipykernel_78271/1055050788.py:14: DtypeWarning:

Columns (8,25) have mixed types. Specify dtype option on import or set low_memory=False.



In [2]:
final_df.head()
final_df.columns
print(final_df.shape)
print(final_df[['id', 'name']].head())



(159463, 81)
         id                     name
0   58172.0                Spacewar!
1  379437.0           ZeusOfTheCrows
2   56681.0  The Oregon Trail (1971)
3   55012.0                   Empire
4   58174.0           Computer Space


In [18]:
# =========================
#  FULL DASHBOARD v2.1
#  (Official code + comparison panel + guarded main graph + button color toggle)
# =========================
#
#  - Uses Dash 2 / Bootstrap.
#  - Adds a collapsible comparison panel that can hold up to 5 games.
#  - Replaces the treemap/heat-map branch with a cleaner version.
#  - Only one graph renders at a time; main graph is frozen while compare panel is open.
#  - “Compare Games” button turns green when panel is open, blue when closed.
#
#  NOTE ───────────────────────────────────────────────────────────────
#  ▸ This file expects a DataFrame called `final_df` to exist globally,
#    with (at minimum) these columns:
#       id • name • released • rating • added • metacritic
#       youtube_count • twitch_count • genres • background_image
#  ▸ Example load:
#       import pandas as pd
#       final_df = pd.read_parquet("games.parquet")
# ====================================================================

import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import pandas as pd              # Used by callbacks
import plotly.graph_objects as go
import squarify
import statsmodels.api as sm     # Used in trends regression
from dash.exceptions import PreventUpdate

# -------------------------
# Dash app
# -------------------------
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    suppress_callback_exceptions=True
)
app.title = "Dashboard"

# -------------------------
# Helper builders
# -------------------------
def build_top_control(active_tab: str):
    return dbc.ButtonGroup(
        [
            dbc.Button(
                "Games",  id="games-button",  n_clicks=0,
                color="primary" if active_tab == "games"  else "secondary"
            ),
            dbc.Button(
                "Trends", id="trends-button", n_clicks=0,
                color="primary" if active_tab == "trends" else "secondary"
            )
        ],
        id="top_control", style={"width": "100%"}
    )


def build_search_bar():
    """Sidebar search (returns matching names in a dropdown)."""
    return dbc.Card(
        dbc.CardBody(
            [
                dbc.Input(
                    id="search_bar",
                    placeholder="Search...",
                    type="text",
                    debounce=False
                ),
                html.Div(                   # results overlay
                    id="search_results",
                    style={
                        "position": "absolute", "zIndex": 1000,
                        "width": "100%", "backgroundColor": "white",
                        "border": "1px solid #ced4da", "borderRadius": "0.25rem",
                        "boxShadow": "0 2px 6px rgba(0,0,0,0.2)",
                        "maxHeight": "200px", "overflowY": "auto",
                        "marginTop": "2px"
                    }
                )
            ]
        ),
        style={
            "position": "relative",
            "marginTop": "1rem", "marginBottom": "1rem",
            "backgroundColor": "white"
        }
    )


def build_compare_search_bar():
    """Compact search bar inside comparison panel."""
    return dbc.Card(
        dbc.CardBody(
            [
                dbc.Input(
                    id="compare_search_bar",
                    placeholder="Search games to add…",
                    type="text", debounce=False,
                    style={"width": "100%"}
                ),
                html.Div(                  # results overlay
                    id="compare_search_results",
                    style={
                        "position": "absolute", "zIndex": 1000,
                        "width": "100%", "backgroundColor": "white",
                        "border": "1px solid #ced4da", "borderRadius": "0.25rem",
                        "boxShadow": "0 2px 6px rgba(0,0,0,0.2)",
                        "maxHeight": "200px", "overflowY": "auto",
                        "marginTop": "2px"
                    }
                )
            ],
            style={"padding": "0.4rem"}
        ),
        style={
            "width": "220px", "maxWidth": "220px", "backgroundColor": "white",
            "boxShadow": "0 1px 3px rgba(0,0,0,0.2)",
            "border": "1px solid #ced4da", "borderRadius": "0.5rem"
        }
    )


def build_games_middle(selected_sort_options: list):
    sort_options = ["Added", "Rating", "YouTube", "Twitch", "Metacritic"]

    sort_buttons = [
        dbc.Checkbox(
            label=s,
            id={"type": "sort-button", "index": s},
            value=s in selected_sort_options,
            style={"marginBottom": "0.4rem"}
        )
        for s in sort_options
    ]

    return html.Div([
        html.H6("Sort By"),
        *sort_buttons,
        html.Div(
            [
                html.Label("Select Genre(s)", style={"fontWeight": "bold", "marginTop": "1rem"}),
                dcc.Dropdown(
                    id={"type": "genre-dropdown", "context": "games"},
                    options=[], multi=True,
                    placeholder="Filter by genre..."
                )
            ],
            style={"marginTop": "1rem"}
        )
    ])



def build_data_view():
    """Main graph + hidden comparison panel inside a flex column."""
    return html.Div(
        [
            html.Div(
                id="main_graph",
                style={
                    "minHeight": 0,
                    "position": "relative",
                    "width": "100%",
                    "overflow": "hidden",
                    "transition": "flex 0.3s ease"
                }
            ),
            html.Div(
                id="comparison_panel",
                style={
                    "display": "none",
                    "overflowY": "auto",
                    "borderTop": "1px solid #dee2e6",
                    "backgroundColor": "#f8f9fa",
                    "transition": "flex 0.3s ease"
                }
            )
        ],
        style={
            "display": "flex",
            "flexDirection": "column",
            "width": "100%",
            "height": "100%"
        }
    )

def build_sidebar(
    active_tab: str,
    selected_sort_options: list,

):
    # -- Games tab controls -----------------------------------------
    games_controls = html.Div(
        build_games_middle(selected_sort_options),
        style={"display": "block" if active_tab == "games" else "none"}
    )

    # Year range (for games tab)
    games_year_slider = html.Div(
        [
            html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="games-year-range-slider",
                min=1970, max=2024, step=1,
                marks={y: str(y) for y in range(1970, 2025, 10)},
                value=[2000, 2020],
                tooltip={"placement": "bottom", "always_visible": False},
                allowCross=False,
                updatemode="mouseup"
            )
        ],
        style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%",
            "display": "block" if active_tab == "games" else "none"
        }
    )

    # Number of games
    games_num_slider = html.Div(
        [
            html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Slider(
                id="games-num-games-slider",
                min=3, max=200, step=1, value=50,
                marks={i: str(i) for i in range(10, 201, 30)},
                updatemode="drag"
            )
        ],
        style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%",
            "display": "block" if active_tab == "games" else "none"
        }
    )

    # -- Trends tab controls ----------------------------------------
    trends_controls = html.Div(
        [
            html.Label("Show games by", style={"fontWeight": "bold", "fontSize": "1.1rem", "marginBottom": "0.5rem"}),
            dbc.Checkbox(id={"type": "trends-metric", "index": "rating"},   label="Rating",      value=True,  style={"marginLeft":"0.25rem","marginBottom":"0.6rem"}),
            dbc.Checkbox(id={"type": "trends-metric", "index": "added"},    label="Added",       value=False, style={"marginLeft":"0.25rem","marginBottom":"0.6rem"}),
            dbc.Checkbox(id={"type": "trends-metric", "index": "metacritic"}, label="Metacritic", value=False, style={"marginLeft":"0.25rem","marginBottom":"0.6rem"}),
            dbc.Checkbox(id={"type": "trends-metric", "index": "youtube_count"}, label="YouTube", value=False, style={"marginLeft":"0.25rem","marginBottom":"0.6rem"}),
            dbc.Checkbox(id={"type": "trends-metric", "index": "twitch_count"},  label="Twitch",   value=False, style={"marginLeft":"0.25rem","marginBottom":"1rem"}),

            html.Label("Other Options", style={"fontWeight": "bold", "fontSize": "1.1rem", "marginTop": "1rem"}),
            dbc.Checkbox(id="show-trendline-checkbox",     value=False, label="Show trendline", style={"marginLeft":"0.25rem","marginBottom":"0.6rem"}),
            dbc.Checkbox(id="show-genre-average-checkbox", value=False, label="Show averages",   style={"marginLeft":"0.25rem","marginBottom":"0.6rem"}),

            html.Div(
                [
                    html.Label("Select Genre(s)", style={"fontWeight":"bold","marginTop":"1rem"}),
                    dcc.Dropdown(
                        id={"type": "genre-dropdown", "context": "trends"}, options=[], multi=True,
                        placeholder="Select genres...", style={"marginBottom":"1rem"}
                    )
                ]
            ),

            html.Div(
                [
                    html.Label("Select Year Range", style={"fontWeight":"bold","marginTop":"1rem"}),
                    dcc.RangeSlider(
                        id="trends-year-range-slider", min=1970, max=2024, step=1,
                        value=[2000, 2020], marks={y: str(y) for y in range(1970, 2025, 10)},
                        tooltip={"placement":"bottom"}
                    ),

                    html.Label("Number of Games", style={"fontWeight":"bold","marginTop":"1.5rem"}),
                    dcc.Slider(
                        id="trends-num-games-slider",
                        min=1000, max=9000, step=500, value=3000,
                        marks={i: str(i) for i in range(1000,9001,2000)},
                        tooltip={"placement":"bottom"}
                    )
                ],
                style={"marginTop":"1rem"}
            )
        ],
        style={"display": "block" if active_tab == "trends" else "none"}
    )

    compare_button = dbc.Button(
        "Compare Games", id="compare-button", n_clicks=0,
        color="secondary",  # always closed by default
        style={"width":"100%", "marginTop":"1rem"}
    )

    return html.Div(
        [
            build_search_bar(),
            html.Hr(),
            build_top_control(active_tab),
            html.Hr(),
            html.Div(
                id="middle_options",
                children=[
                    games_controls,
                    games_year_slider,
                    games_num_slider,
                    trends_controls
                ],
                style={"paddingTop":"1rem","paddingBottom":"1rem"}
            ),
            compare_button
        ],
        style={"padding":"1rem"}
    )


# -------------------------
# Layout
# -------------------------
initial_active_tab   = "games"

app.layout = dbc.Container(
    [
        # Shared stores ------------------------------------------------
        dcc.Store(id="active_main_tab",           data=initial_active_tab),
        dcc.Store(id="selected_sort_options",     data=["Added"]),
        dcc.Store(id="selected_genres",           data=[]),
        dcc.Store(id="selected_year_range",       data=[2000, 2020]),
        dcc.Store(id="selected_game_ids",         data=[]),
        dcc.Store(id="trends_sort_metric",        data="rating"),
        dcc.Store(id="show_trendline",            data=False),
        dcc.Store(id="show_genre_average",        data=False),
        dcc.Store(id="trends_year_range",         data=[2000, 2020]),
        dcc.Store(id="trends_num_games",          data=1000),
        dcc.Store(id="last_clicked_timestamp",    data=None),
        dcc.Store(id="compare_active",            data=False),

        # Grid ---------------------------------------------------------
        dbc.Row(
            [
                dbc.Col(
                    id="sidebar",
                    children=build_sidebar(
                        initial_active_tab,
                        ["Added"],   # initial sort
                    ),
                    width=3,
                    style={
                        "backgroundColor":"#f8f9fa","height":"100vh",
                        "padding":0,"borderRight":"1px solid #dee2e6",
                        "display":"flex","flexDirection":"column"
                    }
                ),
                dbc.Col(
                    build_data_view(),
                    width=9,
                    style={
                        "display":"flex","flexDirection":"column",
                        "height":"100vh","padding":0
                    }
                )
            ]
        )
    ],
    fluid=True
)


# =========================
# Callbacks
# =========================

# -- Main tab / sub-button switching ---------------------------------
@app.callback(
    Output("active_main_tab", "data"),
    [Input("games-button", "n_clicks"),
     Input("trends-button", "n_clicks")],
    State("active_main_tab", "data")
)
def handle_clicks(games_clicks, trends_clicks, current_tab):
    trig = ctx.triggered_id
    if trig == "games-button":
        return "games"
    if trig == "trends-button":
        return "trends"
    if isinstance(trig, dict) and trig.get("type") == "sub-button":
        return current_tab
    return current_tab


# -- Sidebar refresh (includes compare_active for button color) -----
@app.callback(
    Output("sidebar", "children"),
    [
        Input("active_main_tab",        "data"),
        Input("selected_sort_options",  "data"),
    ]
)
def update_sidebar(active_tab, selected_sort_options):
    return build_sidebar(
        active_tab,
        selected_sort_options
    )


# -- Year range (games tab) ------------------------------------------
@app.callback(
    Output("selected_year_range", "data"),
    Input("games-year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


# -- Trend-tab checkboxes/sliders ------------------------------------
@app.callback(Output("show_trendline", "data"),
              Input("show-trendline-checkbox", "value"),
              prevent_initial_call=True)
def upd_trendline(v): return v or False

@app.callback(Output("show_genre_average", "data"),
              Input("show-genre-average-checkbox", "value"))
def upd_avg(v): return v or False

@app.callback(Output("trends_year_range", "data"),
              Input("trends-year-range-slider", "value"))
def upd_trends_year(v): return v

@app.callback(Output("trends_num_games", "data"),
              Input("trends-num-games-slider", "value"))
def upd_trends_num(v): return v


# -- Populate genre dropdown (trends tab) ----------------------------
@app.callback(
    Output({"type": "genre-dropdown", "context": ALL}, "options"),
    Input("active_main_tab", "data")
)
def populate_all_genre_dropdowns(tab):
    genre_set = set()
    for entry in final_df["genres"].dropna():
        if isinstance(entry, list):
            genre_set.update(entry)
        elif isinstance(entry, str):
            genre_set.update([g.strip() for g in entry.split(",")])
    options = [{"label": g, "value": g} for g in sorted(genre_set)]
    return [options] * 2  # games and trends


@app.callback(
    Output("selected_genres", "data"),
    Input({"type": "genre-dropdown", "context": ALL}, "value"),
    State("active_main_tab", "data"),
    prevent_initial_call=True
)
def update_selected_genres_from_dropdown(values, active_tab):
    if active_tab == "games":
        return values[0] if len(values) > 0 else []
    elif active_tab == "trends":
        return values[1] if len(values) > 1 else []
    return []



# -- Enforce single sort selection (games tab) -----------------------
@app.callback(
    [Output({"type": "sort-button", "index": ALL}, "value"),
     Output("selected_sort_options", "data")],
    Input({"type": "sort-button", "index": ALL}, "value"),
    State({"type": "sort-button", "index": ALL}, "id"),
    prevent_initial_call=True
)
def enforce_single_sort(values, ids):
    trig = ctx.triggered_id
    sel = trig["index"] if trig else next((id_["index"] for v, id_ in zip(values, ids) if v), "Added")
    updated = [id_["index"] == sel for id_ in ids]
    return updated, [sel]


# -- Single metric selection (trends tab) ----------------------------
@app.callback(
    [Output({"type": "trends-metric", "index": ALL}, "value"),
     Output("trends_sort_metric", "data")],
    Input({"type": "trends-metric", "index": ALL}, "value"),
    State({"type": "trends-metric", "index": ALL}, "id"),
    prevent_initial_call=True
)
def enforce_single_metric(values, ids):
    trig = ctx.triggered_id
    sel = trig["index"] if trig else next((id_["index"] for v, id_ in zip(values, ids) if v), "rating")
    updated = [id_["index"] == sel for id_ in ids]
    return updated, sel


# -------------------------------------------------------------------
# Sidebar search – returns clickable game links
# -------------------------------------------------------------------
@app.callback(
    Output("search_results", "children"),
    Input("search_bar", "value"),
    prevent_initial_call=True
)
def update_search_results(search_text):
    if not search_text or len(search_text.strip()) < 2:
        return ""
    mask = final_df["name"].str.contains(search_text, case=False, na=False)
    matches = final_df.loc[mask].head(10)
    if matches.empty:
        return html.Small("No matches", style={"color": "#888"})
    return dbc.ListGroup(
        [
            dbc.ListGroupItem(
                html.Span(
                    row["name"],
                    id={"type": "game-link", "index": int(row["id"])},
                    n_clicks=0,
                    style={"color": "#0d6efd", "cursor": "pointer", "textDecoration": "underline"}
                ),
                style={"padding": "0.4rem 0.6rem"}
            )
            for _, row in matches.iterrows()
        ],
        flush=True
    )


# -------------------------------------------------------------------
# Comparison-panel search
# -------------------------------------------------------------------
@app.callback(
    Output("compare_search_results", "children"),
    Input("compare_search_bar", "value"),
    prevent_initial_call=True
)
def update_compare_search_results(search_text):
    if not search_text or len(search_text.strip()) < 2:
        return ""
    # filter your global final_df by case‐insensitive substring
    mask = final_df["name"].str.contains(search_text, case=False, na=False)
    matches = final_df.loc[mask].head(10)
    if matches.empty:
        return html.Small("No matches", style={"color": "#888"})
    return dbc.ListGroup(
        [
            dbc.ListGroupItem(
                html.Span(
                    row["name"],
                    id={"type": "game-link", "index": int(row["id"])},
                    n_clicks=0,
                    style={
                        "color": "#0d6efd",
                        "cursor": "pointer",
                        "textDecoration": "underline"
                    }
                ),
                style={"padding": "0.4rem 0.6rem"}
            )
            for _, row in matches.iterrows()
        ],
        flush=True
    )



# -------------------------------------------------------------------
# Game selection + removal logic
# -------------------------------------------------------------------
@app.callback(
    Output("selected_game_ids", "data"),
    Output("last_clicked_timestamp", "data"),
    Input({"type": "game-link",   "index": ALL}, "n_clicks_timestamp"),
    Input({"type": "remove-game", "index": ALL}, "n_clicks"),
    State({"type": "game-link",   "index": ALL}, "id"),
    State("selected_game_ids",    "data"),
    State("last_clicked_timestamp","data"),
    prevent_initial_call=True
)
def handle_game_selection_and_removal(n_clicks_ts, remove_clicks, ids, selected_ids, last_ts):
    if selected_ids is None:
        selected_ids = []
    trig = ctx.triggered_id
    if isinstance(trig, dict) and trig.get("type") == "remove-game":
        return [gid for gid in selected_ids if gid != trig["index"]], last_ts

    max_ts, clicked = -1, None
    for ts, btn_id in zip(n_clicks_ts, ids):
        if ts and ts > max_ts:
            max_ts, clicked = ts, btn_id["index"]
    if clicked and (last_ts is None or max_ts > last_ts):
        if clicked in selected_ids:
            return selected_ids, max_ts
        if len(selected_ids) == 5:
            selected_ids.pop(0)
        selected_ids.append(clicked)
        return selected_ids, max_ts
    return dash.no_update, dash.no_update


# -------------------------------------------------------------------
# Comparison panel toggle + plot + button color toggle
# -------------------------------------------------------------------
@app.callback(
    Output("compare_active",      "data"),
    Output("selected_game_ids",   "data", allow_duplicate=True),
    Output("comparison_panel",    "children"),
    Output("comparison_panel",    "style"),
    Output("main_graph",          "style"),
    Input("compare-button",       "n_clicks"),
    Input("selected_game_ids",    "data"),
    State("compare_active",       "data"),
    prevent_initial_call=True
)
def update_comparison_panel_and_resize(btn_clicks, game_ids, is_active):
    # ---------- base flex styles --------------------------------------
    base_main_style = {
        "minHeight": 0, "position": "relative", "width": "100%",
        "overflow": "hidden", "transition": "flex 0.3s ease"
    }
    base_compare_style = {
        "overflowY": "auto",                   # panel scrolls
        "borderTop": "1px solid #dee2e6",
        "backgroundColor": "#f8f9fa",
        "transition": "flex 0.3s ease"
    }

    # ---------- toggle logic ------------------------------------------
    trig = dash.callback_context.triggered[0]["prop_id"].split(".")[0]
    if trig == "compare-button" and btn_clicks:
        is_active = not is_active

    if not is_active:
        return (
            False, [], [],                           # compare_active, ids, children
            base_compare_style | {"display": "none"},
            base_main_style   | {"flex": "1 1 0%"}
        )

    # ---------- compact search bar (now its own row) ------------------
    search_bar = build_compare_search_bar()            # plain component, no overlay

    # ---------- placeholder branch ------------------------------------
    if not game_ids:
        placeholder = html.Div(
            "Select games to compare.",
            style={"padding": "2rem", "textAlign": "center",
                   "color": "#666", "fontSize": "1.2rem"}
        )
        panel_children = html.Div(
            [search_bar, placeholder],
            style={"display": "flex", "flexDirection": "column",
                   "width": "100%", "height": "100%"}
        )
        return (
            True, game_ids, panel_children,
            base_compare_style | {"flex": "1.4 1 0%", "display": "block"},
            base_main_style   | {"flex": "2 1 0%"}
        )

    # ---------- data prep ---------------------------------------------
    selected_df = final_df[final_df["id"].isin(game_ids)]
    if selected_df.empty:
        return (
            True, game_ids, [],
            base_compare_style | {"display": "none"},
            base_main_style   | {"flex": "1 1 0%"}
        )

    metric_max = {
        "rating": 5.0,
        "metacritic": 100.0,
        "added": selected_df["added"].max() or 1,
        "youtube_count": selected_df["youtube_count"].max() or 1,
        "twitch_count": selected_df["twitch_count"].max() or 1
    }
    colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]

    # ---------- thumbnail strip (right edge) --------------------------
    thumbs = []
    for idx, gid in enumerate(game_ids):
        row = selected_df[selected_df["id"] == gid].iloc[0]
        thumbs.append(
            html.Div(
                [
                    html.Img(
                        src=row.get("background_image", ""),
                        style={
                            "width": "40px", "height": "40px", "objectFit": "cover",
                            "borderRadius": "50%",
                            "border": f"3px solid {colors[idx % len(colors)]}",
                            "boxShadow": "0 0 3px rgba(0,0,0,0.2)",
                            "cursor": "pointer"
                        },
                        title=row.get("name", "Unnamed Game")
                    ),
                    html.Button(
                        "×",
                        id={"type": "remove-game", "index": gid},
                        style={
                            "position": "absolute", "top": "-5px", "right": "-5px",
                            "border": "none", "background": "red", "color": "white",
                            "borderRadius": "50%", "width": "14px", "height": "14px",
                            "fontSize": "10px", "lineHeight": "12px", "padding": "0",
                            "cursor": "pointer"
                        }
                    )
                ],
                style={"position": "relative", "marginBottom": "0.5rem"}
            )
        )

    thumb_strip = html.Div(
        thumbs,
        style={"display": "flex", "flexDirection": "column", "alignItems": "center",
               "padding": "0.5rem", "borderLeft": "1px solid #ccc",
               "overflowY": "auto", "gap": "0.5rem", "width": "60px"}
    )

    # ---------- histogram ---------------------------------------------
    import plotly.graph_objects as go
    labels = ["Rating", "Metacritic", "Added", "YouTube", "Twitch"]
    fig = go.Figure()

    for idx, gid in enumerate(game_ids):
        row = selected_df[selected_df["id"] == gid].iloc[0]
        raw  = [row.get("rating", 0), row.get("metacritic", 0), row.get("added", 0),
                row.get("youtube_count", 0), row.get("twitch_count", 0)]
        norm = [raw[0]/metric_max["rating"], raw[1]/metric_max["metacritic"],
                raw[2]/metric_max["added"], raw[3]/metric_max["youtube_count"],
                raw[4]/metric_max["twitch_count"]]
        fig.add_trace(
            go.Bar(
                x=labels, y=norm,
                name=row.get("name", f"Game {idx+1}"),
                marker_color=colors[idx % len(colors)],
                marker_line_width=0, opacity=1.0,
                customdata=[f"{v:.1f}" if isinstance(v, (int, float)) else str(v) for v in raw],
                hovertemplate="%{customdata}<extra></extra>"
            )
        )

    fig.update_layout(
        barmode="group",
        xaxis=dict(showline=False, showgrid=False, ticks="", zeroline=False,
                   tickfont=dict(size=12)),
        yaxis=dict(visible=False, range=[0, 1.05], fixedrange=True),
        bargap=0.3, bargroupgap=0.1,
        margin=dict(l=10, r=10, t=10, b=10),
        plot_bgcolor="white",
        showlegend=False
    )

    histogram = html.Div(
        dcc.Graph(figure=fig, style={"height": "100%", "width": "100%"}),
        style={"flex": "1", "padding": "0.5rem",
               "overflow": "hidden", "display": "flex", "alignItems": "center"}
    )

    # ---------- assemble row & panel ----------------------------------
    row = html.Div(
        [histogram, thumb_strip],
        style={"display": "flex", "flexDirection": "row",
               "width": "100%", "height": "100%", "flex": "1 1 0"}
    )

    panel_children = html.Div(
        [search_bar, row],
        style={"display": "flex", "flexDirection": "column",
               "width": "100%", "height": "100%"}
    )

    return (
        True, game_ids, panel_children,
        base_compare_style | {"flex": "1.4 1 0%", "display": "block"},
        base_main_style   | {"flex": "2 1 0%"}
    )


# -------------------------------------------------------------------
# MAIN GRAPH
# -------------------------------------------------------------------
@app.callback(
    Output("main_graph", "children"),
    [
        Input("active_main_tab",       "data"),
        # Games-tab inputs
        Input("selected_sort_options", "data"),
        Input("selected_year_range",   "data"),
        Input("games-num-games-slider","value"),
        # Trends-tab inputs
        Input("trends_year_range",     "data"),
        Input("trends_num_games",      "data"),
        #Input("genre-dropdown",        "data"),
        Input("trends_sort_metric",    "data"),
        Input("show_trendline",        "data"),
        Input("show_genre_average",    "data"),
        Input("selected_genres", "data"),
    ],
)
def update_main_graph(
    active_tab,
    selected_sort_options,
    selected_year_range,
    num_games,
    trends_year_range,
    trends_num_games,
    trends_sort_metric,
    show_trendline,
    show_genre_average,
    selected_genres
):
    import statsmodels.api as sm
    from dash.exceptions import PreventUpdate

    # Only handle our two tabs
    if active_tab not in ("games", "trends"):
        raise PreventUpdate

    # Base dataframe preparation
    df = final_df.copy()
    df = df.dropna(subset=["name"]).drop_duplicates()
    df["release_year"] = pd.to_datetime(df["released"], errors="coerce").dt.year

    # -------------------- TRENDS TAB ---------------------------------
    if active_tab == "trends":
        selected_genres = selected_genres or []
        df = df[
            (df["release_year"] >= trends_year_range[0]) &
            (df["release_year"] <= trends_year_range[1])
        ].dropna(subset=["released", "rating", "genres"])
        df["released"] = pd.to_datetime(df["released"], errors="coerce")
        df = df[df["rating"] > 0.5]

        # filter by genre if requested
        if selected_genres:
            def extract_matching_genre(genre_str):
                try:
                    for g in [g_.strip() for g_ in genre_str.split(",")]:
                        if g in selected_genres:
                            return g
                except Exception:
                    pass
                return None
            df["matched_genre"] = df["genres"].apply(extract_matching_genre)
            df = df[df["matched_genre"].notna()]
            color_col = "matched_genre"
        else:
            df["matched_genre"] = "All"
            color_col = None

        # sample & sort
        df = df.sample(n=min(trends_num_games, len(df)), random_state=42)
        df = df.sort_values("released")

        # metric setup
        y_metric = trends_sort_metric.lower()
        y_label = {
            "rating": "Rating",
            "added": "Added by Users",
            "metacritic": "Metacritic Score",
            "youtube_count": "YouTube Mentions",
            "twitch_count": "Twitch Mentions"
        }.get(y_metric, y_metric.capitalize())
        df[y_metric] = pd.to_numeric(df[y_metric], errors="coerce")
        df = df.dropna(subset=["released", y_metric])

        x_ord = df["released"].map(pd.Timestamp.toordinal)
        y_val = df[y_metric]

        fig = go.Figure()

        # genre-average lines or raw scatter
        if show_genre_average:
            df["year"] = df["released"].dt.year
            grouped = df.groupby(["year", "matched_genre"])[y_metric].mean().reset_index()
            for genre in grouped["matched_genre"].unique():
                gdf = grouped[grouped["matched_genre"] == genre]
                fig.add_trace(go.Scatter(
                    x=gdf["year"], y=gdf[y_metric],
                    mode="lines+markers", name=genre,
                    marker=dict(size=6, opacity=0.8),
                    line=dict(width=2)
                ))
        else:
            for genre in df["matched_genre"].unique():
                gdf = df[df["matched_genre"] == genre]
                fig.add_trace(go.Scatter(
                    x=gdf["released"], y=gdf[y_metric],
                    mode="markers", name=genre,
                    text=gdf["name"],
                    marker=dict(size=6, opacity=0.6)
                ))

        # optional trendline
        if show_trendline and len(x_ord) > 1:
            X = sm.add_constant(x_ord)
            model = sm.OLS(y_val, X).fit()
            trend_y = model.predict(X)
            fig.add_trace(go.Scatter(
                x=df["released"], y=trend_y,
                mode="lines", name="Trendline",
                line=dict(color="orange", width=2)
            ))

        # layout
        y_min, y_max = y_val.min(), y_val.max()
        padding = (y_max - y_min) * 0.05 if y_max > y_min else 1
        fig.update_layout(
            title=f"Game {y_label} Over Time" + (" by Genre" if selected_genres else ""),
            xaxis_title="Release Date",
            yaxis_title=y_label,
            yaxis_range=[max(0, y_min - padding), y_max + padding],
            transition_duration=500
        )

        return dcc.Graph(figure=fig, style={"height": "100%", "width": "100%"})

    # -------------------- GAMES TAB ---------------------------------

    if active_tab == "games":
        mapping = {
            "rating":     "rating",
            "youtube":    "youtube_count",
            "twitch":     "twitch_count",
            "added":      "added",
            "metacritic": "metacritic"
        }
        sort_by = mapping.get(
            (selected_sort_options[0] if selected_sort_options else "rating").lower(),
            "rating"
        )
        # filter by genre if selected
        if selected_genres:
            def has_matching_genre(genre_str):
                try:
                    for g in [g_.strip() for g_ in genre_str.split(",")]:
                        if g in selected_genres:
                            return True
                except Exception:
                    pass
                return False
            df = df[df["genres"].apply(has_matching_genre)]

        # sort and pick top N
        df = (df.dropna(subset=[sort_by])
                .sort_values(sort_by, ascending=False)
                .head(num_games or 50))

        metric = df[sort_by].astype(float).clip(lower=1e-6)
        normed = squarify.normalize_sizes(metric, 100, 100)
        rects  = squarify.squarify(normed, 0, 0, 100, 100)
        df     = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

        tiles = []
        for i ,(_, row) in enumerate(df.iterrows(), start=1):
            tiles.append(
                html.Div(
                    [
                        html.Div(f"{i}. {row["name"]}", style={
                            "fontSize": "12px", "fontWeight": "bold",
                            "overflow": "hidden", "textOverflow": "ellipsis", "whiteSpace": "nowrap"
                        }),
                        html.Div(f"{sort_by.capitalize()}: {row[sort_by]:.2f}",
                                 style={"fontSize": "10px"})
                    ],
                    style={
                        "position": "absolute",
                        "left":   f"{row['x']:.2f}%", "top": f"{row['y']:.2f}%",
                        "width":  f"{row['dx']:.2f}%", "height": f"{row['dy']:.2f}%",
                        "backgroundImage":  f"url('{row['background_image']}')",
                        "backgroundSize":   "cover", "backgroundPosition": "center",
                        "border": "1px solid #fff", "boxSizing": "border-box",
                        "borderRadius": "4px", "overflow": "hidden", "color": "#fff",
                        "display": "flex",
                        "justifyContent": "center",
                        "alignItems": "center",
                        "textAlign": "center",
                        "flexDirection": "column",
                        "padding": "4px"
                    }
                )
            )

        return html.Div(tiles, style={
            "position": "relative", "width": "100%", "height": "100%", "backgroundColor": "#333"
        })

    # Should never get here
    raise PreventUpdate


# =========================
# Run server  (dynamic port)
# =========================
import socket

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))          
    port = s.getsockname()[1]
    s.close()
    return port

PORT = get_free_port()        

def open_browser():
    webbrowser.open_new(f"http://127.0.0.1:{PORT}")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False, port=PORT)

